# Notebook 3: Monte Carlo VaR & Expected Shortfall
**Author:** Niraj Neupane | github.com/nirajneupane17
**Simulations:** 10,000 correlated paths using GBM

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
import warnings; warnings.filterwarnings('ignore')
import sys; sys.path.insert(0, '../src')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize':(13,6),'font.size':11,'axes.titlesize':13,'axes.titleweight':'bold'})

returns = pd.read_csv('../data/returns.csv', index_col='Date', parse_dates=True)
weights = np.array([0.25, 0.25, 0.20, 0.15, 0.15])
port = returns @ weights
print(f'Loaded {len(port):,} observations | {returns.index[0].date()} to {returns.index[-1].date()}')


Loaded 2,609 observations | 2015-01-01 to 2024-12-31


In [2]:
from var_models import monte_carlo_var
np.random.seed(42)
mc = monte_carlo_var(returns, weights, n_sims=10000)
print('Monte Carlo VaR and ES:')
print(mc.round(4))
mc.to_csv('../results/04_monte_carlo_var_es.csv')

Monte Carlo VaR and ES:
                  MC_VaR   MC_ES  ES_VaR_ratio
confidence_level                              
95%               0.0124  0.0155         1.250
99%               0.0174  0.0201         1.155
99%               0.0192  0.0219         1.138


In [3]:
np.random.seed(42)
sim_port = np.random.multivariate_normal(returns.mean().values,returns.cov().values,10000) @ weights

fig,axes=plt.subplots(1,2,figsize=(14,5))
ax1=axes[0]
ax1.hist(sim_port,bins=100,color='#185FA5',alpha=0.7,edgecolor='white',linewidth=0.2,density=True)
for cl,c in zip([0.95,0.99,0.995],['#F57C00','#E24B4A','#7B1FA2']):
    v=abs(np.percentile(sim_port,(1-cl)*100))
    ax1.axvline(-v,color=c,linewidth=1.8,linestyle='--',label=f'{int(cl*100)}% VaR={v:.4f}')
ax1.set_title('Monte Carlo Simulated Distribution (10,000 paths)'); ax1.legend(fontsize=8)

ax2=axes[1]
sizes=[100,500,1000,2000,5000,10000]
conv=[-np.percentile(sim_port[:n],1) for n in sizes]
ax2.plot(sizes,conv,'o-',color='#E24B4A',linewidth=2,markersize=7)
ax2.axhline(conv[-1],color='gray',linewidth=1,linestyle='--',alpha=0.6)
ax2.set_title('99% VaR Convergence'); ax2.set_xscale('log')
plt.tight_layout(); plt.savefig('../results/04_monte_carlo_var.png',dpi=150,bbox_inches='tight')
plt.show()